<a href="https://colab.research.google.com/github/sajidruetcse21/Language-Invariant-Emotion-Detection-Using-Contrastive-Learning/blob/main/Base_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# CELL 1
# Install required libraries for training and evaluation
# ==========================================================

!pip install transformers scikit-learn pandas openpyxl seaborn

In [ ]:
# ==========================================================
# CELL 2
# Import all required Python libraries
# ==========================================================

import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import AdamW

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# ==========================================================
# CELL 3
# Check whether GPU is available
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# ==========================================================
# CELL 4
# Load dataset splits from CSV files
# ==========================================================

train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")
test_df = pd.read_csv("/content/test.csv")

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

train_df.head()

In [ ]:
# ==========================================================
# CELL 5
# Select which language column will be used for training
# and which will be used for testing
# ==========================================================

TRAIN_TEXT_COLUMN = "Bengali"
TEST_TEXT_COLUMN = "Bengali"

LABEL_COLUMN = "Label"

In [ ]:
# ==========================================================
# CELL 6
# Convert emotion labels into numerical form
# ==========================================================

label_encoder = LabelEncoder()

train_df[LABEL_COLUMN] = label_encoder.fit_transform(train_df[LABEL_COLUMN])
val_df[LABEL_COLUMN] = label_encoder.transform(val_df[LABEL_COLUMN])
test_df[LABEL_COLUMN] = label_encoder.transform(test_df[LABEL_COLUMN])

num_labels = len(label_encoder.classes_)
print("Number of emotion classes:", num_labels)
print("Classes:", label_encoder.classes_)

In [ ]:
# ==========================================================
# CELL 7
# Load tokenizer for multilingual transformer
# ==========================================================

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

In [ ]:
# ==========================================================
# CELL 8
# Custom Dataset class for tokenizing text and loading labels
# ==========================================================

class EmotionDataset(Dataset):

    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        item = {key: val.squeeze() for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx])

        return item

In [ ]:
# ==========================================================
# CELL 9
# Create dataset objects for train, validation, and test
# ==========================================================

train_dataset = EmotionDataset(
    train_df[TRAIN_TEXT_COLUMN].tolist(),
    train_df[LABEL_COLUMN].tolist()
)

val_dataset = EmotionDataset(
    val_df[TRAIN_TEXT_COLUMN].tolist(),
    val_df[LABEL_COLUMN].tolist()
)

test_dataset = EmotionDataset(
    test_df[TEST_TEXT_COLUMN].tolist(),
    test_df[LABEL_COLUMN].tolist()
)

In [ ]:
# ==========================================================
# CELL 9 (MODIFIED FOR MULTILINGUAL TRAINING)
# Create datasets using Bangla + Banglish + English together
# ==========================================================

# # ----- TRAIN SET -----

# train_texts = (
#     train_df["Bengali"].tolist() +
#     train_df["Banglish"].tolist() +
#     train_df["English"].tolist()
# )

# train_labels = (
#     train_df[LABEL_COLUMN].tolist() +
#     train_df[LABEL_COLUMN].tolist() +
#     train_df[LABEL_COLUMN].tolist()
# )

# train_dataset = EmotionDataset(train_texts, train_labels)


# # ----- VALIDATION SET -----

# val_texts = (
#     val_df["Bengali"].tolist() +
#     val_df["Banglish"].tolist() +
#     val_df["English"].tolist()
# )

# val_labels = (
#     val_df[LABEL_COLUMN].tolist() +
#     val_df[LABEL_COLUMN].tolist() +
#     val_df[LABEL_COLUMN].tolist()
# )

# val_dataset = EmotionDataset(val_texts, val_labels)


# # ----- TEST SET -----

# test_texts = (
#     test_df["Bengali"].tolist() +
#     test_df["Banglish"].tolist() +
#     test_df["English"].tolist()
# )

# test_labels = (
#     test_df[LABEL_COLUMN].tolist() +
#     test_df[LABEL_COLUMN].tolist() +
#     test_df[LABEL_COLUMN].tolist()
# )

# test_dataset = EmotionDataset(test_texts, test_labels)

In [ ]:
# ==========================================================
# CELL 10
# Create DataLoaders for batching during training
# ==========================================================

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [ ]:
# ==========================================================
# CELL 11
# Load multilingual transformer model for classification
# ==========================================================

model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=num_labels
)

model.to(device)

In [ ]:
# ==========================================================
# CELL 12
# Initialize AdamW optimizer
# ==========================================================

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
# ==========================================================
# CELL 13
# Function to train the model for one epoch
# ==========================================================

def train_epoch():

    model.train()
    total_loss = 0

    for batch in train_loader:

        batch = {k:v.to(device) for k,v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [ ]:
# ==========================================================
# CELL 14
# Evaluate model performance using accuracy and macro F1
# ==========================================================

def evaluate(loader):

    model.eval()

    preds = []
    labels = []

    with torch.no_grad():

        for batch in loader:

            batch = {k:v.to(device) for k,v in batch.items()}

            outputs = model(**batch)

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(labels, preds)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro"
    )

    return acc, precision, recall, f1

In [ ]:
# ==========================================================
# CELL 15
# Training loop for multiple epochs
# ==========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    train_loss = train_epoch()

    val_acc, val_p, val_r, val_f1 = evaluate(val_loader)

    print(f"Epoch {epoch+1}")
    print("Train Loss:", train_loss)
    print("Validation Accuracy:", val_acc)
    print("Validation Macro F1:", val_f1)

In [ ]:
# ==========================================================
# CELL 16
# Evaluate model on the test dataset
# ==========================================================

test_acc, test_p, test_r, test_f1 = evaluate(test_loader)

print("TEST RESULTS")
print("Accuracy:", test_acc)
print("Precision:", test_p)
print("Recall:", test_r)
print("Macro F1:", test_f1)

In [ ]:
# ==========================================================
# CELL 17
# Generate and display confusion matrix for test predictions
# ==========================================================

def plot_confusion_matrix(loader):

    model.eval()

    preds = []
    labels = []

    with torch.no_grad():

        for batch in loader:

            batch = {k:v.to(device) for k,v in batch.items()}

            outputs = model(**batch)

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())

    cm = confusion_matrix(labels, preds)

    class_names = label_encoder.classes_

    plt.figure(figsize=(8,6))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")

    plt.show()

In [ ]:
# ==========================================================
# CELL 18
# Call function to visualize confusion matrix
# ==========================================================

plot_confusion_matrix(test_loader)

In [ ]:
# ==========================================================
# CELL: Generate and Save High-Resolution Confusion Matrix
# ==========================================================

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Emotion label names (change if needed)
label_names = list(label_encoder.classes_)

# Compute confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

# Plot confusion matrix
plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_names,
    yticklabels=label_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

# Save high resolution image
plt.savefig("confusion_matrix.png", dpi=600, bbox_inches="tight")

plt.show()

print("Confusion matrix saved as confusion_matrix.png")